In [ ]:
# ============================================================
# PHASE 4 - SHAP EXPLAINABILITY ANALYSIS
# ============================================================
 
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
import shap
import matplotlib
matplotlib.use('Agg')   
import matplotlib.pyplot as plt
import logging
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from art.attacks.evasion import ProjectedGradientDescent
from art.estimators.classification import TensorFlowV2Classifier
 
logging.getLogger('art').setLevel(logging.ERROR)
 
# Save path — all plots saved in current directory
SHAP_PATH = ""   
if SHAP_PATH and not os.path.exists(SHAP_PATH):
    os.makedirs(SHAP_PATH)

In [ ]:
# ─────────────────────────────────────────────────────────────
# FEATURE NAMES — must match Phase 1 columns exactly
# ─────────────────────────────────────────────────────────────
feature_names = np.array([
    'Header_Length', 'Protocol Type', 'Time_To_Live', 'Rate',
    'fin_flag_number', 'syn_flag_number', 'rst_flag_number',
    'psh_flag_number', 'ack_flag_number', 'ece_flag_number',
    'cwr_flag_number', 'ack_count', 'syn_count', 'fin_count',
    'rst_count', 'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP',
    'SSH', 'IRC', 'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP', 'IGMP',
    'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG', 'Std',
    'Tot size', 'IAT', 'Number', 'Variance'
])
 
# LOAD
le     = joblib.load('label_encoder.pkl')
X_test = np.load('X_test.npy')
y_test = np.load('y_test.npy')

print(f"Full test set: {X_test.shape}")
print(f"Features: {len(feature_names)}")
print(f"Classes:  {le.classes_}")

# SAMPLE 1,000 PER CLASS
SAMPLES_PER_CLASS = 1100
np.random.seed(42)
subset_indices = []

for class_idx in np.unique(y_test):
    class_indices = np.where(y_test == class_idx)[0]
    n_samples     = min(SAMPLES_PER_CLASS, len(class_indices))
    chosen        = np.random.choice(class_indices, size=n_samples, replace=False)
    subset_indices.extend(chosen)

np.random.shuffle(subset_indices)
X_subset = X_test[subset_indices]
y_subset = y_test[subset_indices]

print(f"Sampled subset: {X_subset.shape[0]} samples ({SAMPLES_PER_CLASS} per class)")

input_dim   = X_subset.shape[1]
num_classes = len(np.unique(y_test))

# SPLIT
X_temp, X_test_held, y_temp, y_test_held = train_test_split(
    X_subset, y_subset,
    test_size=0.25,
    random_state=42,
    stratify=y_subset
)
X_tr, X_v, y_tr, y_v = train_test_split(
    X_temp, y_temp,
    test_size=0.167,
    random_state=42,
    stratify=y_temp
)

print(f"Train:  {X_tr.shape}")
print(f"Test:   {X_test_held.shape}")

In [51]:
# ─────────────────────────────────────────────────────────────
# MODEL BUILDER — matches Phase 1 architecture (256-128-64-32)
# ─────────────────────────────────────────────────────────────
def build_model(input_dim, num_classes, label_smoothing=False):
    inputs = tf.keras.Input(shape=(input_dim,))
    x = tf.keras.layers.Dense(256, activation='relu')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(32, activation='relu')(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)
 
    model = tf.keras.Model(inputs, outputs)
    loss  = (
        tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
        if label_smoothing
        else 'sparse_categorical_crossentropy'
    )
    model.compile(optimizer='adam', loss=loss, metrics=['accuracy'])
    return model

In [ ]:
# ─────────────────────────────────────────────────────────────
# LOAD MODELS
# ─────────────────────────────────────────────────────────────
# Baseline — load full model directly from Phase 1
print("\nLoading baseline model...")
baseline_model = tf.keras.models.load_model('baseline_model.keras')
 
# Adversarially trained — build architecture then load weights
print("Loading adversarial model...")
adv_model = build_model(input_dim, num_classes, label_smoothing=True)
adv_model.set_weights(baseline_model.get_weights())  
adv_model.load_weights('adv_model_seed0.weights.h5') 
 
# Verify both models
baseline_acc = accuracy_score(
    y_test_held,
    baseline_model.predict(X_test_held, verbose=0).argmax(axis=1)
)
adv_acc = accuracy_score(
    y_test_held,
    adv_model.predict(X_test_held, verbose=0).argmax(axis=1)
)
print(f"\nBaseline accuracy:    {baseline_acc:.4f}")
print(f"Adversarial accuracy: {adv_acc:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# BUILD BACKGROUND & EXPLANATION SETS
# ─────────────────────────────────────────────────────────────
np.random.seed(42)
 
background_indices = []
for class_idx in np.unique(y_tr):
    class_mask = np.where(y_tr == class_idx)[0]
    chosen     = np.random.choice(class_mask, size=min(25, len(class_mask)), replace=False)
    background_indices.extend(chosen)
 
X_background = X_tr[background_indices].astype(np.float32)
print(f"\nBackground set: {X_background.shape}  (25 per class × 8 = 200 samples)")
 
explain_indices = []
for class_idx in np.unique(y_test_held):
    class_mask = np.where(y_test_held == class_idx)[0]
    chosen     = np.random.choice(class_mask, size=min(50, len(class_mask)), replace=False)
    explain_indices.extend(chosen)
 
X_explain = X_test_held[explain_indices].astype(np.float32)
y_explain = y_test_held[explain_indices]
print(f"Explanation set: {X_explain.shape}  (50 per class × 8 = 400 samples)")

In [ ]:
# ─────────────────────────────────────────────────────────────
# GENERATE ADVERSARIAL EXAMPLES FOR EXPLANATION
# ─────────────────────────────────────────────────────────────
loss_fn    = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
classifier = TensorFlowV2Classifier(
    model=baseline_model,
    nb_classes=num_classes,
    input_shape=(input_dim,),
    loss_object=loss_fn,
    clip_values=(float(X_explain.min()), float(X_explain.max()))
)
 
pgd = ProjectedGradientDescent(
    estimator=classifier,
    eps=0.1,
    eps_step=0.01,
    max_iter=10,
    num_random_init=3
)
 
X_explain_adv = pgd.generate(x=X_explain, batch_size=256)
print(f"Adversarial examples generated: {X_explain_adv.shape}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# SHAP ANALYSIS — 4 combinations
# ─────────────────────────────────────────────────────────────
print("\n[1/4] Baseline model — clean data...")
explainer_baseline    = shap.DeepExplainer(baseline_model, X_background)
shap_baseline_clean   = explainer_baseline.shap_values(X_explain)
 
print("\n[2/4] Baseline model — adversarial data...")
shap_baseline_adv     = explainer_baseline.shap_values(X_explain_adv)
 
print("\n[3/4] Adversarial model — clean data...")
explainer_adv         = shap.DeepExplainer(adv_model, X_background)
shap_adv_clean        = explainer_adv.shap_values(X_explain)
 
print("\n[4/4] Adversarial model — adversarial data...")
shap_adv_adv          = explainer_adv.shap_values(X_explain_adv)
 
 
# ─────────────────────────────────────────────────────────────
# SAVE SHAP VALUES
# ─────────────────────────────────────────────────────────────
np.save(f'{SHAP_PATH}shap_baseline_clean.npy', shap_baseline_clean)
np.save(f'{SHAP_PATH}shap_baseline_adv.npy',   shap_baseline_adv)
np.save(f'{SHAP_PATH}shap_adv_clean.npy',      shap_adv_clean)
np.save(f'{SHAP_PATH}shap_adv_adv.npy',        shap_adv_adv)
np.save(f'{SHAP_PATH}X_explain.npy',           X_explain)
np.save(f'{SHAP_PATH}X_explain_adv.npy',       X_explain_adv)
np.save(f'{SHAP_PATH}y_explain.npy',           y_explain)
print("\nSHAP values saved.")
 
 
# ─────────────────────────────────────────────────────────────
# HELPER
# ─────────────────────────────────────────────────────────────
def get_mean_abs_shap(shap_values):
    """Average absolute SHAP values across all classes."""
    return np.mean([np.abs(sv) for sv in shap_values], axis=0)
 
shap_baseline_clean_mean = get_mean_abs_shap(shap_baseline_clean)
shap_baseline_adv_mean   = get_mean_abs_shap(shap_baseline_adv)
shap_adv_clean_mean      = get_mean_abs_shap(shap_adv_clean)
shap_adv_adv_mean        = get_mean_abs_shap(shap_adv_adv)
 

In [ ]:
# ─────────────────────────────────────────────────────────────
# PLOTS
# ─────────────────────────────────────────────────────────────
 
# Plot 1: Baseline model — clean data
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_baseline_clean, X_explain,
    feature_names=feature_names,
    class_names=list(le.classes_),
    show=False
)
plt.title("Baseline Model — Feature Importance (Clean Data)")
plt.tight_layout()
plt.savefig(f'{SHAP_PATH}plot1_baseline_clean.png', dpi=150)
plt.close()
print("Plot 1 saved: Baseline model clean data")
 
# Plot 2: Baseline model — adversarial data
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_baseline_adv, X_explain_adv,
    feature_names=feature_names,
    class_names=list(le.classes_),
    show=False
)
plt.title("Baseline Model — Feature Importance (Adversarial Data)")
plt.tight_layout()
plt.savefig(f'{SHAP_PATH}plot2_baseline_adv.png', dpi=150)
plt.close()
print("Plot 2 saved: Baseline model adversarial data")
 
# Plot 3: Adversarial model — clean data
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_adv_clean, X_explain,
    feature_names=feature_names,
    class_names=list(le.classes_),
    show=False
)
plt.title("Defended Model — Feature Importance (Clean Data)")
plt.tight_layout()
plt.savefig(f'{SHAP_PATH}plot3_adv_clean.png', dpi=150)
plt.close()
print("Plot 3 saved: Defended model clean data")
 
# Plot 4: Adversarial model — adversarial data
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_adv_adv, X_explain_adv,
    feature_names=feature_names,
    class_names=list(le.classes_),
    show=False
)
plt.title("Defended Model — Feature Importance (Adversarial Data)")
plt.tight_layout()
plt.savefig(f'{SHAP_PATH}plot4_adv_adv.png', dpi=150)
plt.close()
print("Plot 4 saved: Defended model adversarial data")
 
# Plot 5: Feature vulnerability comparison
feature_importance_clean = np.mean(shap_baseline_clean_mean, axis=0)
feature_importance_adv   = np.mean(shap_baseline_adv_mean,   axis=0)
vulnerability            = np.abs(feature_importance_adv - feature_importance_clean)
 
sorted_idx   = np.argsort(vulnerability)[::-1][:15]
sorted_names = [feature_names[i] for i in sorted_idx]
sorted_vuln  = vulnerability[sorted_idx]
 
plt.figure(figsize=(10, 6))
plt.barh(sorted_names[::-1], sorted_vuln[::-1], color='crimson')
plt.xlabel('Feature Importance Shift (Clean → Adversarial)')
plt.title('Top 15 Most Vulnerable Features\n(Baseline Model)')
plt.tight_layout()
plt.savefig(f'{SHAP_PATH}plot5_vulnerability.png', dpi=150)
plt.close()
print("Plot 5 saved: Feature vulnerability ranking")
 
# Plot 6: Defense stability comparison
baseline_shift       = np.abs(feature_importance_adv - feature_importance_clean)
adv_importance_clean = np.mean(shap_adv_clean_mean, axis=0)
adv_importance_adv   = np.mean(shap_adv_adv_mean,   axis=0)
defended_shift       = np.abs(adv_importance_adv - adv_importance_clean)
 
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].barh(sorted_names[::-1], baseline_shift[sorted_idx][::-1], color='crimson')
axes[0].set_title('Baseline Model\nFeature Importance Shift')
axes[0].set_xlabel('Importance Shift')
axes[1].barh(sorted_names[::-1], defended_shift[sorted_idx][::-1], color='steelblue')
axes[1].set_title('Defended Model\nFeature Importance Shift')
axes[1].set_xlabel('Importance Shift')
plt.suptitle('Defense Stability: Feature Importance Shift Under Attack', fontsize=13)
plt.tight_layout()
plt.savefig(f'{SHAP_PATH}plot6_defense_stability.png', dpi=150)
plt.close()
print("Plot 6 saved: Defense stability comparison")
 

In [ ]:
# ─────────────────────────────────────────────────────────────
# SUMMARY STATISTICS
# ─────────────────────────────────────────────────────────────
print("\n" + "="*50)
print("PHASE 4 SUMMARY")
print("="*50)
 
print("\nTop 10 most important features (baseline, clean data):")
top_features = np.argsort(feature_importance_clean)[::-1][:10]
for rank, idx in enumerate(top_features, 1):
    print(f"  {rank}. {feature_names[idx]:<20} {feature_importance_clean[idx]:.4f}")
 
print("\nTop 10 most vulnerable features:")
top_vulnerable = np.argsort(vulnerability)[::-1][:10]
for rank, idx in enumerate(top_vulnerable, 1):
    print(f"  {rank}. {feature_names[idx]:<20} shift: {vulnerability[idx]:.4f}")
 
print("\nDefense stability:")
print(f"  Avg feature shift (baseline): {np.mean(baseline_shift):.4f}")
print(f"  Avg feature shift (defended): {np.mean(defended_shift):.4f}")
reduction = (
    (np.mean(baseline_shift) - np.mean(defended_shift))
    / np.mean(baseline_shift) * 100
)
print(f"  Stability improvement:        {reduction:.1f}%")
 
print(f"\nAll plots saved to: '{SHAP_PATH if SHAP_PATH else 'current directory'}'")
print("\nPhase 4 complete!")